# 00 イントロ：Qwen3-4B とチャットしてみる

このノートブックでは **Qwen3-4B** を動かしながら、言語モデルの基本を体験します。

## このノートブックでやること

1. **環境確認** — デバイス (CPU / MPS / CUDA) とパッケージを確認する
2. **モデル読み込み** — Hugging Face から Tokenizer と Model を準備する
3. **チャット** — シングルターン・マルチターンで会話する
4. **Thinking モード** — 推論過程を `<think>` ブロックで観察する
5. **chat template** — special token がどこに入るかを確認する（参考）

## Qwen3-4B とは

- Alibaba が開発した **4B（40億）パラメータ**の言語モデル
- 中国語・英語・日本語など多言語対応
- **Thinking モード**（推論を段階的に考える）と **non-thinking モード** を切り替えられる
- 基本は non-thinking モード（`enable_thinking=False`）で使い、Section 4 で Thinking モードも試す

## 全体の流れ（Transformer の処理）

```
テキスト
  ↓ tokenizer.apply_chat_template()  # チャット形式のフォーマットに変換
トークン列
  ↓ model.generate()                 # Transformer が次のトークンを繰り返し予測
生成トークン列
  ↓ tokenizer.decode()               # トークンをテキストに戻す
応答テキスト
```

## 0. 環境セットアップ（3環境 自動切替: Colabだけ pip / path も自動）


In [1]:
# 3環境(Mac/Win/Colab)を同一ファイルで動かすための判定。Colabのみ pip（Mac/Winはenvに在るのでskip）。
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "transformers==5.9.0" "accelerate==1.13.0"
    print("Colab: pip done")
else:
    print("local(Mac/Win): pip skip（env利用）")

local(Mac/Win): pip skip（env利用）


---
## 1. 環境確認

In [2]:
import sys
import logging
import torch
import transformers

# HuggingFace Hub の認証警告を抑制する（cache から読む場合は不要なため）
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print("device:", device)
print("dtype:", dtype)

Python: 3.12.10 (main, Jun 28 2026, 04:07:27) [Clang 21.0.0 (clang-2100.1.1.101)]
PyTorch: 2.12.1
Transformers: 5.9.0
device: mps
dtype: torch.float16


---
## 2. モデル読み込み

Tokenizer と Model を Hugging Face cache から読み込みます。  
初回は自動でダウンロードされます（約 8 GB）。  
2回目以降は cache から読むので高速です。

### トークナイザー

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

MODEL_ID = "Qwen/Qwen3-4B"

print("tokenizer 読み込み中...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("  語彙サイズ:", tokenizer.vocab_size)
print("  tokenizer クラス:", type(tokenizer).__name__)

tokenizer 読み込み中...


  語彙サイズ: 151643
  tokenizer クラス: Qwen2Tokenizer


**トークナイザーの動作を少しだけ見てみる**

In [4]:
ids = tokenizer.encode("京都大学の情報学科")
print("token ids:", ids)
for i, tid in enumerate(ids):
    piece = tokenizer.decode([tid]) 
    print(f"{i:2d}: {tid:6d} -> '{piece}'")

token ids: [115806, 99562, 15767, 134481, 104391]
 0: 115806 -> '京都'
 1:  99562 -> '大学'
 2:  15767 -> 'の'
 3: 134481 -> '情報'
 4: 104391 -> '学科'


### モデル

In [5]:
print("model 読み込み中...")
model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    attn_implementation="eager",  # attention weights を取得できる実装
)
model.to(device)  # pyright: ignore[reportArgumentType]
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"  パラメータ数: {total_params / 1e9:.2f}B")
print(f"  モデルクラス: {type(model).__name__}")
print(f"  レイヤー数: {model.config.num_hidden_layers}")
print(f"  隠れ層の次元数: {model.config.hidden_size}")
print(f"  attention head 数: {model.config.num_attention_heads}")

model 読み込み中...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  パラメータ数: 4.02B
  モデルクラス: Qwen3ForCausalLM
  レイヤー数: 36
  隠れ層の次元数: 2560
  attention head 数: 32


**ちょっとだけモデルを見てみる**

In [6]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

**Qwen3-4B アーキテクチャ 補足**

| 行 | 説明 |
|---|---|
| `Embedding(151936, 2560)` | 語彙 151,936 トークンを 2,560 次元ベクトルに変換 |
| `36 x Qwen3DecoderLayer` | 36 層の Transformer ブロックを積み重ねる |
| `Qwen3Attention` | Self-Attention：各トークンが全トークンを参照して文脈を把握 |
| `q_proj: Linear(2560→4096)` | Query 射影：32 heads × 128 次元 |
| `k_proj: Linear(2560→1024)` | Key 射影：8 heads × 128 次元（GQA — Query より head 数が少ない） |
| `v_proj: Linear(2560→1024)` | Value 射影：8 heads × 128 次元（GQA） |
| `o_proj: Linear(4096→2560)` | Attention 出力を元の 2,560 次元に戻す |
| `q_norm / k_norm` | Query・Key に対する RMS 正規化（QK Norm） |
| `Qwen3MLP` | Feed-Forward Network（FFN）：SwiGLU 活性化を使用 |
| `gate_proj / up_proj: Linear(2560→9728)` | SwiGLU の 2 つの入力線形層（中間次元 9,728） |
| `down_proj: Linear(9728→2560)` | FFN 出力を 2,560 次元に戻す |
| `input_layernorm` | Attention 前の RMS 正規化（Pre-Norm） |
| `post_attention_layernorm` | FFN 前の RMS 正規化（Pre-Norm） |
| `norm: Qwen3RMSNorm((2560,))` | 全層通過後の最終 RMS 正規化 |
| `rotary_emb: Qwen3RotaryEmbedding` | 位置エンコーディング（RoPE） |
| `lm_head: Linear(2560→151936)` | 隠れ状態 → 語彙分布（次トークンの確率を出力） |


### アーキテクチャを深堀りする

`inspect.getsource()` を使うと、pip install 版の Transformers でもソースコードを確認できます。

In [7]:
import inspect
from transformers.models.qwen3.modeling_qwen3 import (
    Qwen3DecoderLayer, Qwen3Attention, Qwen3MLP
)

**各層の処理順（Qwen3DecoderLayer.forward）**

In [8]:
print(inspect.getsource(Qwen3DecoderLayer.forward))

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = False,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        # Self Attention
        hidden_states, _ = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        hidden_states = residual + hidden_states

        # Fully Connected
        residual = hidden_states
        hidden_states = self.post_att

**アテンション（Qwen3Attention）**

In [9]:
print(inspect.getsource(Qwen3Attention))

@use_kernelized_func(apply_rotary_pos_emb)
class Qwen3Attention(nn.Module):
    """Multi-headed attention from 'Attention Is All You Need' paper"""

    def __init__(self, config: Qwen3Config, layer_idx: int):
        super().__init__()
        self.layer_type = config.layer_types[layer_idx] if hasattr(config, "layer_types") else None
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
        self.num_key_value_groups = config.num_attention_heads // config.num_key_value_heads
        self.scaling = self.head_dim**-0.5
        self.attention_dropout = config.attention_dropout
        self.is_causal = True

        self.q_proj = nn.Linear(
            config.hidden_size, config.num_attention_heads * self.head_dim, bias=config.attention_bias
        )
        self.k_proj = nn.Linear(
            config.hidden_size, config.num_key_value_heads * self.head_dim, bias=config.atten

**FFN（Qwen3MLP）**

In [10]:
print(inspect.getsource(Qwen3MLP))

class Qwen3MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj



---

## 3. チャット

### チャット関数

`chat()` 関数を定義します。

処理の流れ：
1. `messages`（ユーザーとアシスタントのやりとりリスト）を **chat template** でフォーマット
2. テキストを **トークン列**（整数の配列）に変換
3. `model.generate()` で次のトークンを繰り返し予測 → 応答トークン列
4. 生成部分だけを取り出してテキストにデコード

**`messages` の構造（`list[dict]`）**

`messages` は Python の「リスト」で、各要素が「辞書（dict）」になっています。

```python
# list：[ ] で囲まれた並び
# dict：{ } で囲まれたキーと値のペア

messages = [                                     # list の開始
    {"role": "user",      "content": "こんにちは"},  # dict（1つ目）
    {"role": "assistant", "content": "こんにちは！"}, # dict（2つ目）
    {"role": "user",      "content": "今日の天気は？"},# dict（3つ目）
]                                                # list の終了
```

各 `dict` には必ず2つのキーがあります：

| キー | 値 | 意味 |
|---|---|---|
| `"role"` | `"user"` または `"assistant"` | 誰の発言か |
| `"content"` | 任意のテキスト | 発言の内容 |

会話のターンが増えるたびに、この `list` に `dict` が追加されていきます。

In [11]:
def chat(messages: list[dict],
            max_new_tokens: int = 256, enable_thinking: bool = False, skip_special_tokens: bool = True) -> str:
    """messages を受け取り、モデルの応答テキストを返す。"""
    # chat template を適用してプロンプト文字列を作る
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    # トークン化
    inputs = tokenizer(text, return_tensors="pt").to(device)

    # 生成
    with torch.no_grad():
        output_ids = model.generate(  # type: ignore[union-attr]
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # 入力部分を除いた生成トークンだけをデコード
    n_input = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][n_input:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=skip_special_tokens)
    return response

### シングルターン

1 往復のシンプルな会話から始めます。

In [12]:
# シングルターンの例
user_text = "言語モデルとは何か、1〜2文で教えてください。"
messages = [
    {"role": "user", "content": user_text}
]

response = chat(messages)
print("[User]", user_text)
print("[Assistant]", response)

[User] 言語モデルとは何か、1〜2文で教えてください。
[Assistant] 言語モデルは、文を生成したり、文の意味を理解したりするための人工知能です。


### マルチターン

会話履歴（`messages` リスト）に応答を追加し続けることで、**文脈を保ったやりとり**ができます。

Transformer は入力トークン列全体を毎回処理するため、
過去の発言をリストに入れて渡すだけで文脈が維持されます。

In [13]:
def chat_turn(messages: list[dict], user_text: str, **kwargs) -> list[dict]:
    """user_text を追加して応答を得る。更新済み messages を返す。"""
    messages = messages + [{"role": "user", "content": user_text}]
    response = chat(messages, **kwargs)
    messages = messages + [{"role": "assistant", "content": response}]
    return messages


def print_turn(history: list[dict]):
    print(f"[User]      {history[-2]['content']}")
    print(f"[Assistant] {history[-1]['content']}")
    print()

In [14]:
# 1ターン目: 最初の質問
history = []
history = chat_turn(history, "トークンとは何ですか？")
print_turn(history)

[User]      トークンとは何ですか？
[Assistant] トークン（Token）とは、コンピュータ科学やプログラミング、特にプログラミング言語やコンパイラの文法解析において、**コードやテキストを小さな単位に分割したものです**。

---

### トークンの基本的な意味

トークンは、プログラムやテキストを解析する際に、**意味のある最小単位**として扱われる単語や記号です。例えば、プログラミング言語のコードを解析する際には、`if`、`for`、`while`、`int`、`return`などのキーワードや、`+`、`-`、`*`、`/`などの演算子、変数名や関数名など、すべてがトークンとして処理されます。

---

### トークンの種類

トークンにはいくつかの種類があります：

1. **キーワード（Keyword）**  
   プログラム言語で使われる特殊な単語。  
   例：`if`, `for`, `while`, `int`, `return`, `function`, `class` など。





In [15]:
# 2ターン目：前の回答を受けて続ける
history = chat_turn(history, "具体的に「京都」という単語はいくつのトークンに分割されますか？")
print_turn(history)

[User]      具体的に「京都」という単語はいくつのトークンに分割されますか？
[Assistant] 「京都」という単語は、**1つのトークン**として扱われます。

---

### 理由

トークンは、**意味のある最小単位**を指します。  
「京都」は、単語としての意味を持ち、語彙的にも一つの単語です。  
したがって、**トークン解析（トークナイズ）**の過程で、「京都」は**1つのトークン**として分割されます。

---

### 例

例えば、以下のような文を考えます：

> 「京都の観光地は、多くの人が訪れます。」

この文をトークン化すると、以下のように分割されます：

- 京都
- の
- 観光地
- は
- 、
- 多くの
- 人が
- 訪れます
- 。

このように、**「京都」は1つのトークン**として扱われています。

---

### まとめ

- 「京都」は、**1つのトークン**として扱われます。
- トークンは、意味のある最小単位であり、単語や記号などが



In [16]:
# 3ターン目
history = chat_turn(history, "ありがとう。では Qwen3 の語彙サイズはいくつですか？")
print_turn(history)

[User]      ありがとう。では Qwen3 の語彙サイズはいくつですか？
[Assistant] ありがとう！  
では、**Qwen3** の語彙サイズ（ vocabulary size ）について説明します。

---

### Qwen3 の語彙サイズ

**Qwen3** は、通称で「Qwen3」と呼ばれるモデルで、**175000語（175k）** の語彙を含んでいます。

---

### 語彙サイズとは？

語彙サイズ（vocabulary size）とは、モデルが理解・生成できる**単語や記号の総数**を指します。  
これは、モデルが何種類の単語や表現を処理できるかを示す指標です。

---

### Qwen3 の語彙サイズの意味

- **175000語**：Qwen3 は、約17万語の単語や表現を処理できます。
- これは、一般的な言語（例：日本語、英語など）の語彙量に比べて、**非常に豊富**です。
- これにより、Qwen3 は幅広いトピックや文脈を理解



**ハルシネーションの確認**

モデルは「175,000語」と答えましたが、正しいでしょうか？`tokenizer.vocab_size` で実際の値を確認してみます。

In [17]:
actual = tokenizer.vocab_size
print(f"実際の語彙サイズ: {actual:,}")

実際の語彙サイズ: 151,643


---

## 4. Thinking モード

Qwen3 には通常の応答モードに加えて、**Thinking モード**があります。
モデルが回答する前に `<think>...</think>` ブロックの中で推論過程を展開します。

`enable_thinking=True` に変えるだけで有効になります。
thinking 部分が長くなるため、`max_new_tokens` は大きめに設定します。

In [18]:
user_text = "言語モデルとは何か、1〜2文で教えてください。"
messages = [{"role": "user", "content": user_text}]

response = chat(
    messages,
    max_new_tokens=512,
    enable_thinking=True,
    skip_special_tokens=False,  # <think> タグを見るため
)
print(response)

<think>
Okay, the user is asking for a definition of a language model in 1-2 sentences. Let me start by recalling what a language model is. It's a type of AI that understands and generates human language. I need to mention its purpose, like processing text and generating responses. Also, maybe include that it's based on machine learning, particularly deep learning. But keep it concise. Let me check if I need to specify the technology, like neural networks. Maybe not necessary for a brief definition. Focus on the main function: understanding and generating text. Also, mention that it's used for tasks like answering questions, writing, etc. But stay within two sentences. Let me try to put that together.
</think>

言語モデルは、人間の言語を理解し、文を生成する人工知能であり、自然言語処理や質問への回答、文章の作成など、さまざまなタスクに応用されます。機械学習特に深層学習を基盤としており、大量のテキストデータからパターンを学習して言語の意味や文脈を把握します。<|im_end|>


`skip_special_tokens=False` にしているため、通常は非表示になる special token も出力に含まれます。

| 出力 | 意味 |
|---|---|
| `<think>...</think>` | モデルの推論過程（Thinking モード） |
| `<\|im_end\|>` | アシスタントの発言終了を示す special token |

通常の chat（`skip_special_tokens=True`）では `<\|im_end\|>` は自動的に除去されます。

---

## 5. 参考：chat template の中身を見る

`apply_chat_template()` が何を作っているか確認します。
Special token（`<|im_start|>` など）がどこに入るかを見てみましょう。

In [19]:
sample_messages = [
    {"role": "user", "content": "こんにちは"},
    {"role": "assistant", "content": "こんにちは！何かお手伝いできますか？"},
    {"role": "user", "content": "今日の天気は？"},
]

formatted = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(formatted)

<|im_start|>user
こんにちは<|im_end|>
<|im_start|>assistant
こんにちは！何かお手伝いできますか？<|im_end|>
<|im_start|>user
今日の天気は？<|im_end|>
<|im_start|>assistant
<think>

</think>




In [20]:
# トークン列も見てみる
token_ids = tokenizer.encode(formatted)
print(f"トークン数: {len(token_ids)}")
print()

for i, tid in enumerate(token_ids):
    piece = tokenizer.decode([tid])
    print(f"  [{i:2d}] id={tid:6d}  '{piece}'")

トークン数: 39

  [ 0] id=151644  '<|im_start|>'
  [ 1] id=   872  'user'
  [ 2] id=   198  '
'
  [ 3] id= 89015  'こんにちは'
  [ 4] id=151645  '<|im_end|>'
  [ 5] id=   198  '
'
  [ 6] id=151644  '<|im_start|>'
  [ 7] id= 77091  'assistant'
  [ 8] id=   198  '
'
  [ 9] id= 89015  'こんにちは'
  [10] id=  6313  '！'
  [11] id=130967  '何か'
  [12] id= 32234  'お'
  [13] id= 44934  '手'
  [14] id=132322  '伝'
  [15] id= 16586  'い'
  [16] id=130225  'できます'
  [17] id= 31049  'か'
  [18] id= 11319  '？'
  [19] id=151645  '<|im_end|>'
  [20] id=   198  '
'
  [21] id=151644  '<|im_start|>'
  [22] id=   872  'user'
  [23] id=   198  '
'
  [24] id=102242  '今日'
  [25] id= 15767  'の'
  [26] id= 35727  '天'
  [27] id= 94121  '気'
  [28] id= 15322  'は'
  [29] id= 11319  '？'
  [30] id=151645  '<|im_end|>'
  [31] id=   198  '
'
  [32] id=151644  '<|im_start|>'
  [33] id= 77091  'assistant'
  [34] id=   198  '
'
  [35] id=151667  '<think>'
  [36] id=   271  '

'
  [37] id=151668  '</think>'
  [38] id=   271  '

'


---
## まとめ

| ステップ | 処理 | 関数 |
|---|---|---|
| フォーマット | テキスト → chat template 形式 | `tokenizer.apply_chat_template()` |
| トークン化 | テキスト → token ID 列 | `tokenizer()` |
| 生成 | token ID 列 → 応答 token ID 列 | `model.generate()` |
| デコード | 応答 token ID 列 → テキスト | `tokenizer.decode()` |

次のノートブックでは **tokenizer** の動作を詳しく観察します。